## 1. Download Dataset from Kaggle

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rituparnaghosh18/transformed-housing-data-2")
print("Path to dataset files:", path)

## 2. Load & Inspect Data

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os, glob

sns.set_theme(style='whitegrid', palette='Blues_d')

# Find CSV in downloaded path
csv_files = glob.glob(os.path.join(path, '*.csv'))
print('Files found:', csv_files)

df = pd.read_csv(csv_files[0])
print(f'Shape: {df.shape}')
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
# Missing values
print(df.isnull().sum()[df.isnull().sum() > 0])

## 3. Data Preprocessing

In [ ]:
# Standardise column names
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace(r'[^\w]', '_', regex=True)

# Drop duplicates
df.drop_duplicates(inplace=True)

# Fill missing numeric values with median
for col in df.select_dtypes(include=np.number).columns:
    df[col].fillna(df[col].median(), inplace=True)

CURRENT_YEAR = 2024

# Derived columns
if 'yr_built' in df.columns:
    df['house_age'] = CURRENT_YEAR - df['yr_built']

if 'yr_renovated' in df.columns:
    df['renovated']          = df['yr_renovated'].apply(lambda x: 'Renovated' if x > 0 else 'Not Renovated')
    df['yr_since_renovation'] = df['yr_renovated'].apply(lambda x: CURRENT_YEAR - x if x > 0 else CURRENT_YEAR - df['yr_built'].mean())

if 'price' in df.columns:
    df['price_bin'] = pd.cut(df['price'],
        bins=[0,200000,400000,600000,800000,1000000,float('inf')],
        labels=['<200K','200K-400K','400K-600K','600K-800K','800K-1M','>1M'])

if 'house_age' in df.columns:
    df['age_group'] = pd.cut(df['house_age'],
        bins=[0,10,20,30,50,75,float('inf')],
        labels=['0-10 yrs','11-20 yrs','21-30 yrs','31-50 yrs','51-75 yrs','75+ yrs'])

print('Preprocessing complete. Shape:', df.shape)
df.head()

## 4. Scenario 1 — Overall Data Overview (KPIs)

In [ ]:
print(f"Total Records        : {len(df):,}")
if 'price' in df.columns:        print(f"Average Sale Price   : ${df['price'].mean():,.2f}")
if 'sqft_basement' in df.columns: print(f"Total Basement Sqft  : {df['sqft_basement'].sum():,.0f}")

## 5. Scenario 2 — Total Sales by Years Since Renovation

In [ ]:
if {'yr_since_renovation','price_bin'}.issubset(df.columns):
    grp = df.groupby('price_bin', observed=True)['yr_since_renovation'].sum().reset_index()
    grp.columns = ['Price Bin', 'Total Yrs Since Renovation']

    fig, ax = plt.subplots(figsize=(10,5))
    sns.barplot(data=grp, x='Price Bin', y='Total Yrs Since Renovation', palette='Blues_r', ax=ax)
    ax.set_title('Scenario 2 — Total Sales by Years Since Renovation', fontsize=14, fontweight='bold')
    ax.set_xlabel('Sale Price Bin')
    ax.set_ylabel('Total Years Since Renovation')
    plt.tight_layout()
    plt.show()

## 6. Scenario 3 — House Age Distribution by Renovation Status

In [ ]:
if 'age_group' in df.columns:
    pie_data = df['age_group'].value_counts(sort=False)
    colors   = sns.color_palette('Blues', n_colors=len(pie_data))

    fig, ax = plt.subplots(figsize=(7,7))
    ax.pie(pie_data, labels=pie_data.index, autopct='%1.1f%%',
           colors=colors, startangle=140, wedgeprops={'edgecolor':'white','linewidth':1.5})
    ax.set_title('Scenario 3 — House Age Distribution by Renovation Status', fontsize=13, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 7. Scenario 4 — House Age by Bathrooms, Bedrooms & Floors

In [ ]:
feature_cols = [c for c in ['bathrooms','bedrooms','floors'] if c in df.columns]

if 'age_group' in df.columns and feature_cols:
    fig, axes = plt.subplots(1, len(feature_cols), figsize=(6*len(feature_cols), 6))
    if len(feature_cols)==1: axes=[axes]

    for ax, feat in zip(axes, feature_cols):
        top_vals = df[feat].value_counts().head(5).index
        grp4 = df[df[feat].isin(top_vals)].groupby(['age_group', feat], observed=True).size().reset_index(name='count')
        sns.barplot(data=grp4, x='age_group', y='count', hue=feat, palette='Blues', ax=ax)
        ax.set_title(f'Age Group vs {feat.capitalize()}', fontweight='bold')
        ax.tick_params(axis='x', rotation=30)
        ax.legend(title=feat.capitalize(), fontsize=8)

    fig.suptitle('Scenario 4 — House Age Distribution by Features', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

## 8. Export CSVs for Tableau

In [ ]:
import os
EXPORT = '../data/tableau_exports'
os.makedirs(EXPORT, exist_ok=True)

df.to_csv(f'{EXPORT}/housing_full_cleaned.csv', index=False)
print('Exported full cleaned dataset.')

if {'yr_since_renovation','price_bin','price'}.issubset(df.columns):
    scen2 = df.groupby('price_bin', observed=True).agg(
        total_sales=('price','sum'),
        avg_yr_since_renovation=('yr_since_renovation','mean'),
        house_count=('price','count')
    ).reset_index()
    scen2.to_csv(f'{EXPORT}/scenario2_sales_by_renovation.csv', index=False)
    print('Exported Scenario 2.')

if {'age_group','renovated'}.issubset(df.columns):
    scen3 = df.groupby(['age_group','renovated'], observed=True).size().reset_index(name='count')
    scen3.to_csv(f'{EXPORT}/scenario3_age_renovation.csv', index=False)
    print('Exported Scenario 3.')

for feat in [c for c in ['bathrooms','bedrooms','floors'] if c in df.columns]:
    scen4 = df.groupby(['age_group', feat], observed=True).size().reset_index(name='count')
    scen4.to_csv(f'{EXPORT}/scenario4_age_by_{feat}.csv', index=False)
    print(f'Exported Scenario 4 ({feat}).')

print('\nAll exports done!')